In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_16_try_2.pickle

In [ ]:
%%RecordEvent
%%time
### cell 17 ###

# Consolidate and GPU-accelerate the per-year percentage computation
question = question_of_interest_cell29
xcol = title_for_x_axis_cell29  # this is '' in our case
subset = subset_of_degrees
# single replace mapping for all years
replacements = {
    "Masterâ\x80\x99s degree": "Master's degree",
    "Bachelorâ\x80\x99s degree": "Bachelor's degree"
}

# map each DataFrame to its year, rename 2017's column locally
df_map = {
    '2017': responses_df_2017.rename(columns={'FormalEducation': question}),
    '2018': responses_df_2018_cell10,
    '2019': responses_df_2019_cell10,
    '2020': responses_df_2020,
    '2021': responses_df_2021,
    '2022': responses_df_2022_cell10
}

df_list = []
for year, df_src in df_map.items():
    # clean the column values via a single .replace call
    df_clean = df_src.assign(**{question: df_src[question].replace(replacements)})
    # GPU-accelerated groupby + size to get counts (including nulls)
    df_counts = df_clean.groupby(question, dropna=False).size().reset_index(name='count')
    total = df_counts['count'].sum()
    # compute percentage and attach year
    df_counts = df_counts.assign(
        percentage=(df_counts['count'] * 100 / total).round(1),
        year=year
    ).drop(columns='count')
    # rename the question column to the x-axis title (blank string)
    df_counts = df_counts.rename(columns={question: xcol})
    # collect in standardized column order
    df_list.append(df_counts[['percentage', 'year', xcol]])

# concatenate all years and sort on GPU
degree_df_combined_cell29 = pd.concat(df_list, ignore_index=True)
degree_df_combined_cell29 = degree_df_combined_cell29.sort_values(
    by=['year', 'percentage'], ascending=[True, True]
)
# map all other categories to 'Other' in one vectorized step
degree_df_combined_cell29[xcol] = degree_df_combined_cell29[xcol].where(
    degree_df_combined_cell29[xcol].isin(subset),
    'Other'
)
# inspect
degree_df_combined_cell29.info()

In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_17_try_4.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_17_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[17], f)


In [ ]:
opt_output = Out.get(4)